# Cycles et demi-cycles du global workspace

Mêmes définitions que les losses de shimmer :
- **demi-cycle** : domaine → GW → même domaine (image → GW → image, digit → GW → digit) ;
- **cycle** : domaine → GW → autre domaine → GW → domaine de départ (image → digit → image, digit → image → digit).

Les losses sont calculées comme à l'entraînement (`compute_loss` des domaines, dans l'espace latent),
puis complétées par des mesures plus lisibles : MSE en pixels, précision sur le chiffre, erreur d'angle.
Tout est évalué sur le test set de MNIST, en images droites et tournées d'angles connus.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from cfg_tools import load_config_files

from numb_project.constants import BASE, CONFIG_FILE, CONFIG_FOLDER, DEVICE, GW_CHECKPOINT_FOLDER
from numb_project.data_module import MnistDataModule
from numb_project.utils import angle_to_cos_sin, rotate_images
from numb_project.domain_module import get_domains_config
from numb_project.gw_model import MyGlobalWorkspace, get_global_workspace_mods

CKPT = f"{GW_CHECKPOINT_FOLDER}/rotation_test.ckpt"

torch.set_grad_enabled(False)  # notebook d'évaluation uniquement

config = load_config_files(CONFIG_FOLDER, use_cli=False, load_files=[CONFIG_FILE])[0]
domain_configs = get_domains_config(['image', 'digit'], config['domains']['digit'])
gw_mod, selection_mod, loss_mod, operation_rotate_pixel_mod, operation_rotate_gw_mod = get_global_workspace_mods(config, domain_configs)

gw = MyGlobalWorkspace.load_from_checkpoint(
    CKPT,
    gw_mod=gw_mod,
    selection_mod=selection_mod,
    loss_mod=loss_mod,
    operation_rotate_pixel_mod=operation_rotate_pixel_mod,
    operation_rotate_gw_mod=operation_rotate_gw_mod,
    map_location=DEVICE,
    weights_only=False,
)
gw.eval()
device = gw.device

image_mod = gw.gw_mod.domain_mods["image"]
digit_mod = gw.gw_mod.domain_mods["digit"]
mnist_vae = image_mod.vae

In [ ]:
# --- Passages entre les espaces (latents des domaines <-> état GW) ---
def to_gw(domain: str, latent: torch.Tensor) -> torch.Tensor:
    # État GW réel : tanh de l'encodage GW (cf GWModule.fuse)
    return gw.gw_mod.fusion_activation_fn(gw.gw_mod.gw_encoders[domain](latent))

def from_gw(domain: str, gw_state: torch.Tensor) -> torch.Tensor:
    return gw.gw_mod.gw_decoders[domain](gw_state)

def demi_cycle(domain: str, latent: torch.Tensor) -> torch.Tensor:
    return from_gw(domain, to_gw(domain, latent))

def cycle(domain: str, through: str, latent: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Renvoie (latent intermédiaire dans `through`, latent revenu dans `domain`)."""
    intermediate = from_gw(through, to_gw(domain, latent))
    return intermediate, from_gw(domain, to_gw(through, intermediate))

def digit_vector(labels: torch.Tensor, angles_deg: torch.Tensor) -> torch.Tensor:
    return torch.cat([F.one_hot(labels, BASE).float(), angle_to_cos_sin(angles_deg)], dim=-1)

def read_angle(digits: torch.Tensor) -> torch.Tensor:
    return torch.rad2deg(torch.atan2(digits[:, BASE + 1], digits[:, BASE])) % 360

def angle_error(pred_deg: torch.Tensor, target_deg: torch.Tensor) -> torch.Tensor:
    return ((pred_deg - target_deg + 180) % 360 - 180).abs()

def digit_scores(pred: torch.Tensor, labels: torch.Tensor, angles: torch.Tensor) -> str:
    accuracy = (pred[:, :BASE].argmax(dim=-1) == labels).float().mean()
    err = angle_error(read_angle(pred), angles)
    return f"précision chiffre = {accuracy:6.1%}, erreur d'angle moyenne = {err.mean():5.1f}° (médiane {err.median():5.1f}°)"

def show_rows(rows: list[tuple[str, torch.Tensor]], n_show: int = 8) -> None:
    fig, axes = plt.subplots(len(rows), n_show, figsize=(n_show * 1.2, 1.3 * len(rows)), squeeze=False)
    for r, (label, images) in enumerate(rows):
        for i in range(n_show):
            axes[r, i].imshow(images[i, 0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
            axes[r, i].set_xticks([])
            axes[r, i].set_yticks([])
        axes[r, 0].set_ylabel(label, fontsize=8, rotation=0, ha="right", va="center")
    plt.tight_layout()
    plt.show()


# --- Données : un batch du test set, droit et tourné d'angles connus ---
data_module = MnistDataModule(512)
batch, _, _ = next(iter(data_module.val_dataloader()))
pair = batch[frozenset({"digit", "image"})]
upright_images = pair["image"].to(device)
labels = pair["digit"][:, :BASE].argmax(dim=-1).to(device)

generator = torch.Generator(device=device).manual_seed(0)
rotation_angles = torch.rand(len(labels), device=device, generator=generator) * 360
rotated_images = rotate_images(upright_images, rotation_angles)

# (nom, images, angles, digits) ; le digit contient l'angle de l'image
datasets_to_test = [
    ("droites", upright_images, torch.zeros_like(rotation_angles)),
    ("tournées", rotated_images, rotation_angles),
]
datasets_to_test = [(name, images, angles, digit_vector(labels, angles)) for name, images, angles in datasets_to_test]
print("batch :", upright_images.shape)

## 1. Demi-cycles

In [ ]:
# --- Demi-cycle image : image -> GW -> image ---
# loss d'entraînement = MSE sur le latent du VAE ; en pixels, le VAE seul est le plancher.
for name, images, _, _ in datasets_to_test:
    latent = image_mod.encode(images)
    latent_back = demi_cycle("image", latent)
    loss = image_mod.compute_loss(latent_back, latent, images).loss
    mse_gw = F.mse_loss(image_mod.decode(latent_back), images)
    mse_vae = F.mse_loss(image_mod.decode(latent), images)
    print(f"images {name:<8}: loss (latent) = {loss:.4f}   MSE pixels via GW = {mse_gw:.5f}   (VAE seul = {mse_vae:.5f})")

latent = image_mod.encode(rotated_images)
show_rows([
    ("original", rotated_images),
    ("VAE seul", image_mod.decode(latent)),
    ("demi-cycle", image_mod.decode(demi_cycle("image", latent))),
])

In [ ]:
# --- Demi-cycle digit : digit -> GW -> digit ---
# On teste aussi des angles quelconques (le domaine digit seul est entraîné avec des angles aléatoires).
for name, _, angles, digits in datasets_to_test:
    digits_back = demi_cycle("digit", digits)
    loss = digit_mod.compute_loss(digits_back, digits, digits)
    print(f"digits {name:<8}: loss = {loss.loss:.4f} (chiffre {loss.metrics['digit']:.4f}, angle {loss.metrics['angle']:.4f})   "
          + digit_scores(digits_back, labels, angles))

## 2. Cycles

In [ ]:
# --- Cycle image -> digit -> image ---
# Le digit ne garde que le chiffre et l'angle : le style d'écriture est perdu,
# l'image revenue est un chiffre "type" au bon angle. Le digit intermédiaire
# est l'équivalent de la traduction image -> digit.
for name, images, angles, _ in datasets_to_test:
    latent = image_mod.encode(images)
    digits_mid, latent_back = cycle("image", "digit", latent)
    loss = image_mod.compute_loss(latent_back, latent, images).loss
    mse = F.mse_loss(image_mod.decode(latent_back), images)
    print(f"images {name:<8}: loss (latent) = {loss:.4f}   MSE pixels = {mse:.5f}   digit intermédiaire : "
          + digit_scores(digits_mid, labels, angles))

latent = image_mod.encode(rotated_images)
digits_mid, latent_back = cycle("image", "digit", latent)
show_rows([
    ("original", rotated_images),
    ("demi-cycle", image_mod.decode(demi_cycle("image", latent))),
    ("cycle", image_mod.decode(latent_back)),
])

In [ ]:
# --- Cycle digit -> image -> digit ---
# Deux variantes :
#  - comme à l'entraînement, l'image intermédiaire reste dans le latent du VAE ;
#  - plus strict, l'image intermédiaire est décodée en pixels puis ré-encodée par le VAE.
for name, _, angles, digits in datasets_to_test:
    latent_mid, digits_back = cycle("digit", "image", digits)
    loss = digit_mod.compute_loss(digits_back, digits, digits)
    digits_back_pixels = from_gw("digit", to_gw("image", image_mod.encode(image_mod.decode(latent_mid))))
    print(f"digits {name:<8}: loss = {loss.loss:.4f} (chiffre {loss.metrics['digit']:.4f}, angle {loss.metrics['angle']:.4f})")
    print(f"    via latent VAE : " + digit_scores(digits_back, labels, angles))
    print(f"    via pixels     : " + digit_scores(digits_back_pixels, labels, angles))

latent_mid, _ = cycle("digit", "image", datasets_to_test[1][3])
show_rows([("image réelle", rotated_images), ("image intermédiaire", image_mod.decode(latent_mid))])

## 3. Qualité selon l'angle

In [ ]:
# --- Erreurs en fonction de l'angle de l'image (par tranches de 30°) ---
latent = image_mod.encode(rotated_images)
demi_cycle_mse = ((image_mod.decode(demi_cycle("image", latent)) - rotated_images) ** 2).mean(dim=(1, 2, 3))
digits_mid, _ = cycle("image", "digit", latent)
image_to_digit_err = angle_error(read_angle(digits_mid), rotation_angles)
image_to_digit_ok = (digits_mid[:, :BASE].argmax(dim=-1) == labels).float()
_, digits_back = cycle("digit", "image", datasets_to_test[1][3])
digit_cycle_err = angle_error(read_angle(digits_back), rotation_angles)

bins = torch.arange(0, 361, 30, device=device)
centers = ((bins[:-1] + bins[1:]) / 2).cpu()
bin_idx = torch.bucketize(rotation_angles, bins[1:-1])

def per_bin(values: torch.Tensor) -> list[float]:
    return [values[bin_idx == b].mean().item() for b in range(len(centers))]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(centers, per_bin(demi_cycle_mse), "-o")
axes[0].set_title("demi-cycle image : MSE pixels")
axes[1].plot(centers, per_bin(image_to_digit_ok), "-o")
axes[1].set_title("image -> digit : précision chiffre")
axes[2].plot(centers, per_bin(image_to_digit_err), "-o", label="image -> digit")
axes[2].plot(centers, per_bin(digit_cycle_err), "-o", label="cycle digit -> image -> digit")
axes[2].set_title("erreur d'angle moyenne (°)")
axes[2].legend()
for ax in axes:
    ax.set_xlabel("angle de l'image (°)")
    ax.set_xticks(range(0, 361, 60))
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()